# Category 4 — DML & String operations
### Total Questions: 6

---

1. Write a SQL query to swap the seat id of every two consecutive students
2. Update the salary column by increasing it by 10%
3. Write a SQL query to delete duplicate rows in a table
4. Write a SQL query to find the numbers which consecutively occur exactly 3 times
5. Write a SQL query to find the numbers which consecutively occur more than 1 time
6. Write a query to get mean, median and mode for salary

⬆️ Upload sql_practice.db from your computer

In [ ]:
# ⬆️ Upload sql_practice.db from your computer
from google.colab import files
uploaded = files.upload()
print("✅ Database uploaded!")

Saving sql_practice.db to sql_practice.db
✅ Database uploaded!


In [ ]:
import sqlite3
import pandas as pd
import os

_cwd = os.getcwd()
_db = os.path.join(_cwd, "sql_practice.db")
if not os.path.isfile(_db):
    _db = os.path.join(os.path.dirname(_cwd), "sql_practice.db")
conn = sqlite3.connect(_db)

# Verify tables loaded
pd.read_sql_query("""
SELECT name FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)


,name
0,customers
1,department
2,emp
3,employee_salary
4,employees
5,monthly_salary
6,num
7,orders
8,posts
9,products


## 1. Swap Seat IDs of Every Two Consecutive Students

**Difficulty:** 🟡 Medium

**Subtopic:** CASE WHEN / Conditional Logic

---

**Table: seat**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| student     | varchar |
+-------------+---------+
```

- `id` represents the seat number.
- `student` represents the student's name.
- If the number of students is **odd**, the last student's ID is not swapped.

---

**Problem**

Write a SQL query to **swap the seat ID of every two consecutive students**.

Return the result ordered by `id` in ascending order.

**Example 1**

**Input**

seat table:
```
+----+---------+
| id | student |
+----+---------+
| 1  | A       |
| 2  | B       |
| 3  | C       |
| 4  | D       |
| 5  | E       |
+----+---------+
```

**Output**
```
+----+---------+
| id | student |
+----+---------+
| 1  | B       |
| 2  | A       |
| 3  | D       |
| 4  | C       |
| 5  | E       |
+----+---------+
```

Preview the table

In [ ]:
# 👀 Preview the table
print("seat table:")
pd.read_sql_query("SELECT * FROM seat", conn)

seat table:


,id,student
0,1,A
1,2,B
2,3,C
3,4,D
4,5,E


✅ Solution-

In [ ]:
# ✅ Solution
query = """
SELECT
    CASE
        WHEN id % 2 = 1 AND id + 1 <= (SELECT MAX(id) FROM seat) THEN id + 1
        WHEN id % 2 = 0 THEN id - 1
        ELSE id
    END AS id,
    student
FROM seat
ORDER BY id
"""

pd.read_sql_query(query, conn)

,id,student
0,1,B
1,2,A
2,3,D
3,4,C
4,5,E


**Explanation**

- `id % 2 = 1` checks if the seat ID is **odd**.
- For odd IDs — swap with the next seat by returning `id + 1`, but only if `id + 1` exists (handles the last odd student).
- For even IDs — swap with the previous seat by returning `id - 1`.
- `ELSE id` handles the last student when total count is odd — their ID stays the same.
- `ORDER BY id` returns the result in ascending seat order.

> 💡 Think of it as — odd seats borrow the next seat's ID, even seats borrow the previous seat's ID.

## 2. UPDATE Salary Column by Increasing It 10%

**Difficulty:** 🟢 Easy

**Subtopic:** DML / UPDATE Statement

---

**Table: employees**
```
+------------+---------+
| Column Name| Type    |
+------------+---------+
| id         | int     |
| name       | varchar |
| salary     | int     |
| dep_id     | int     |
| manager_id | int     |
| hire_date  | date    |
+------------+---------+
```

- `salary` represents the employee's current salary.

---

**Problem**

Write a SQL query to **UPDATE the salary column** by increasing every employee's salary by **10%**.

**Example 1**

**Before UPDATE — employees table:**
```
+----+-------+--------+
| id | name  | salary |
+----+-------+--------+
| 1  | John  | 60000  |
| 2  | Alice | 80000  |
| 3  | Bob   | 75000  |
| 4  | David | 50000  |
+----+-------+--------+
```

**After UPDATE:**
```
+----+-------+--------+
| id | name  | salary |
+----+-------+--------+
| 1  | John  | 66000  |
| 2  | Alice | 88000  |
| 3  | Bob   | 82500  |
| 4  | David | 55000  |
+----+-------+--------+
```

In [ ]:
# 👀 Before UPDATE
print("Before UPDATE:")
pd.read_sql_query("SELECT id, name, salary FROM employees", conn)

Before UPDATE:


,id,name,salary
0,1,John,60000
1,2,Alice,80000
2,3,Bob,75000
3,4,David,50000
4,5,John,45000
5,6,Emma,90000
6,7,Frank,55000
7,8,Grace,72000
8,9,Henry,48000
9,10,Isla,95000


In [ ]:
# ✅ UPDATE salary by 10%
conn.execute("""
UPDATE employees
SET salary = salary * 1.10
""")
conn.commit()

# 👀 After UPDATE
print("After UPDATE:")
pd.read_sql_query("SELECT id, name, salary FROM employees", conn)

After UPDATE:


,id,name,salary
0,1,John,66000.0
1,2,Alice,88000.0
2,3,Bob,82500.0
3,4,David,55000.0
4,5,John,49500.0
5,6,Emma,99000.0
6,7,Frank,60500.0
7,8,Grace,79200.0
8,9,Henry,52800.0
9,10,Isla,104500.0


**Explanation**

- `UPDATE employees` targets the entire employees table.
- `SET salary = salary * 1.10` multiplies every employee's current salary by 1.10 — adding a 10% raise.
- No `WHERE` clause means the update applies to **all rows**.
- We preview the table before and after to clearly see the change.
- To revert: `UPDATE employees SET salary = salary / 1.10`

> 💡 In production always use a `WHERE` clause to avoid accidentally updating all rows — e.g. `WHERE dep_id = 1` to update only one department.

## 3. Delete Duplicate Rows From a Table

**Difficulty:** 🔴 Hard

**Subtopic:** DML / DELETE + Subquery + ROWID

---

**Table: employees**
```
+------------+---------+
| Column Name| Type    |
+------------+---------+
| id         | int     |
| name       | varchar |
| salary     | int     |
| dep_id     | int     |
| manager_id | int     |
| hire_date  | date    |
+------------+---------+
```

---

**Problem**

Write a SQL query to **delete all duplicate rows** from the employees table, keeping only **one copy** of each duplicate.

**Example 1**

**Setup — Insert duplicate rows first:**
```
+----+-------+--------+
| id | name  | salary |
+----+-------+--------+
| 1  | John  | 60000  |
| 2  | Alice | 80000  |
| 2  | Alice | 80000  |  ← duplicate
| 3  | Bob   | 75000  |
| 3  | Bob   | 75000  |  ← duplicate
+----+-------+--------+
```

**After DELETE:**
```
+----+-------+--------+
| id | name  | salary |
+----+-------+--------+
| 1  | John  | 60000  |
| 2  | Alice | 80000  |
| 3  | Bob   | 75000  |
+----+-------+--------+
```

In [ ]:
# 👀 Preview the table
print("employees table:")
pd.read_sql_query("SELECT * FROM employees", conn)

employees table:


,id,name,salary,dep_id,manager_id,hire_date,email
0,1,John,66000.0,1,NaN,2026-01-15,john@gmail.com
1,2,Alice,88000.0,1,1.0,2026-02-01,alice@gmail.com
2,3,Bob,82500.0,2,1.0,2026-01-20,john@gmail.com
3,4,David,55000.0,2,3.0,2026-03-01,bob@gmail.com
4,5,John,49500.0,3,3.0,2026-02-15,david@gmail.com
5,6,Emma,99000.0,1,1.0,2025-06-10,emma@gmail.com
6,7,Frank,60500.0,2,3.0,2025-08-20,frank@gmail.com
7,8,Grace,79200.0,3,5.0,2026-01-05,grace@gmail.com
8,9,Henry,52800.0,3,5.0,2025-11-11,alice@gmail.com
9,10,Isla,104500.0,1,1.0,2025-04-01,isla@gmail.com


In [ ]:
# 🛠️ Insert duplicate rows to demonstrate
conn.execute("""
INSERT INTO employees VALUES (2, 'Alice', 80000, 1, 1, '2023-03-01','alice@gmail.com')
""")
conn.execute("""
INSERT INTO employees VALUES (3, 'Bob', 75000, 2, 1, '2023-05-01','bob@gmail.com')
""")
conn.commit()

# 👀 Before DELETE — showing duplicates
print("Before DELETE — with duplicates:")
pd.read_sql_query("SELECT id, name, salary FROM employees ORDER BY id", conn)

Before DELETE — with duplicates:


,id,name,salary
0,1,John,66000.0
1,2,Alice,88000.0
2,2,Alice,80000.0
3,2,Alice,80000.0
4,3,Bob,82500.0
5,3,Bob,75000.0
6,4,David,55000.0
7,5,John,49500.0
8,6,Emma,99000.0
9,7,Frank,60500.0


In [ ]:
# ✅ Delete duplicate rows keeping the one with lowest rowid
conn.execute("""
DELETE FROM employees
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM employees
    GROUP BY id, name, salary
)
""")
conn.commit()

# 👀 After DELETE
print("After DELETE — duplicates removed:")
pd.read_sql_query("SELECT id, name, salary FROM employees ORDER BY id", conn)

After DELETE — duplicates removed:


,id,name,salary
0,1,John,66000.0
1,2,Alice,88000.0
2,2,Alice,80000.0
3,3,Bob,82500.0
4,3,Bob,75000.0
5,4,David,55000.0
6,5,John,49500.0
7,6,Emma,99000.0
8,7,Frank,60500.0
9,8,Grace,79200.0


**Explanation**

- `rowid` is a hidden unique identifier that SQLite automatically assigns to every row.
- `GROUP BY id, name, salary` groups rows that are exact duplicates.
- `MIN(rowid)` picks the **first occurrence** of each duplicate group to keep.
- `DELETE WHERE rowid NOT IN (...)` deletes all rows **except** the first occurrence.
- This is the cleanest way to remove duplicates in SQLite.

> 💡 In MySQL you would use:
> ```sql
> DELETE e1 FROM employees e1
> INNER JOIN employees e2
> WHERE e1.id = e2.id AND e1.rowid > e2.rowid
> ```

## 4. Find Numbers That Consecutively Occur Exactly 3 Times

**Difficulty:** 🔴 Hard

**Subtopic:** Self JOIN / Consecutive rows logic

---

**Table: num**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| numbers     | int     |
+-------------+---------+
```

- `id` represents the row number (sequential).
- `numbers` represents the value in that row.

---

**Problem**

Write a SQL query to find all numbers that appear **consecutively exactly 3 times** in a row.

**Example 1**

**Input**

num table:
```
+----+---------+
| id | numbers |
+----+---------+
| 1  | 1       |
| 2  | 1       |
| 3  | 1       |
| 4  | 2       |
| 5  | 3       |
| 6  | 3       |
| 7  | 3       |
| 8  | 3       |
+----+---------+
```

**Output**
```
+---------+
| numbers |
+---------+
| 1       |
+---------+
```

> 💡 Number 3 appears 4 consecutive times so it does NOT qualify. Only number 1 appears exactly 3 consecutive times.

In [ ]:
# 👀 Preview the table
print("num table:")
pd.read_sql_query("SELECT * FROM num", conn)

num table:


,id,numbers
0,1,1
1,2,1
2,3,1
3,4,2
4,5,3
5,6,3
6,7,3
7,8,3


In [ ]:
# ✅ Solution
query = """
SELECT DISTINCT n1.numbers
FROM num n1
JOIN num n2 ON n2.id = n1.id + 1
JOIN num n3 ON n3.id = n1.id + 2
WHERE n1.numbers = n2.numbers
  AND n2.numbers = n3.numbers
  AND NOT EXISTS (
      SELECT 1 FROM num n0
      WHERE n0.id = n1.id - 1
        AND n0.numbers = n1.numbers
  )
  AND NOT EXISTS (
      SELECT 1 FROM num n4
      WHERE n4.id = n1.id + 3
        AND n4.numbers = n1.numbers
  )
"""

pd.read_sql_query(query, conn)

,numbers
0,1


**Explanation**

- We join `num` with itself 3 times — `n1`, `n2`, `n3` — to check 3 consecutive rows.
- `n2.id = n1.id + 1` and `n3.id = n1.id + 2` ensures they are consecutive.
- `n1.numbers = n2.numbers = n3.numbers` ensures all 3 values are the same.
- `NOT EXISTS` before and after ensures there is no 4th consecutive occurrence.
  - Checks that `n1.id - 1` does NOT have the same number (no occurrence before).
  - Checks that `n1.id + 3` does NOT have the same number (no occurrence after).
- Number 3 fails because `id = 8` also has 3 — making it 4 consecutive times.

## 5. Find Numbers That Consecutively Occur More Than 1 Time

**Difficulty:** 🟡 Medium

**Subtopic:** Self JOIN / Consecutive rows logic

---

**Table: num**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| numbers     | int     |
+-------------+---------+
```

- `id` represents the row number (sequential).
- `numbers` represents the value in that row.

---

**Problem**

Write a SQL query to find all numbers that appear **consecutively more than once** (at least 2 times in a row).

**Example 1**

**Input**

num table:
```
+----+---------+
| id | numbers |
+----+---------+
| 1  | 1       |
| 2  | 1       |
| 3  | 1       |
| 4  | 2       |
| 5  | 3       |
| 6  | 3       |
| 7  | 3       |
| 8  | 3       |
+----+---------+
```

**Output**
```
+---------+
| numbers |
+---------+
| 1       |
| 3       |
+---------+
```

In [ ]:
# 👀 Preview the table
print("num table:")
pd.read_sql_query("SELECT * FROM num", conn)

num table:


,id,numbers
0,1,1
1,2,1
2,3,1
3,4,2
4,5,3
5,6,3
6,7,3
7,8,3


In [ ]:
# ✅ Solution
query = """
SELECT DISTINCT n1.numbers
FROM num n1
JOIN num n2 ON n2.id = n1.id + 1
WHERE n1.numbers = n2.numbers
"""

pd.read_sql_query(query, conn)

,numbers
0,1
1,3


**Explanation**

- We join `num` with itself — `n1` and `n2`.
- `n2.id = n1.id + 1` ensures `n2` is the **very next row** after `n1`.
- `n1.numbers = n2.numbers` checks that both consecutive rows have the **same value**.
- `DISTINCT` ensures each number appears only once in the output even if it repeats many times.
- Compare with Q4 — Q4 checks for **exactly 3** consecutive, Q5 checks for **at least 2** consecutive.

## 6. Mean, Median and Mode for Salary

**Difficulty:** 🔴 Hard

**Subtopic:** Aggregation / Window Functions / Subquery

---

**Table: emp**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| salary      | int     |
+-------------+---------+
```

- `id` represents the employee ID.
- `salary` represents the employee's salary.

---

**Problem**

Write SQL queries to calculate the **Mean**, **Median** and **Mode** of the salary column.

**Example 1**

**Input**

emp table:
```
+----+--------+
| id | salary |
+----+--------+
| 1  | 10000  |
| 2  | 15000  |
| 3  | 20000  |
| 4  | 20000  |
| 5  | 30000  |
+----+--------+
```

**Output**
```
+----------+--------+-------+
| mean     | median | mode  |
+----------+--------+-------+
| 19000.0  | 20000  | 20000 |
+----------+--------+-------+
```

In [ ]:
# 👀 Preview the table
print("emp table:")
pd.read_sql_query("SELECT * FROM emp", conn)

emp table:


,id,salary
0,1,10000
1,2,15000
2,3,20000
3,4,20000
4,5,30000
5,6,30000
6,7,45000
7,8,50000
8,9,50000
9,10,75000


In [ ]:
# ✅ MEAN
print("Mean:")
pd.read_sql_query("""
SELECT AVG(salary) AS mean
FROM emp
""", conn)

Mean:


,mean
0,34500.0


In [ ]:
# ✅ MEDIAN
print("Median:")
pd.read_sql_query("""
SELECT AVG(salary) AS median
FROM (
    SELECT salary
    FROM (
        SELECT salary,
               ROW_NUMBER() OVER (ORDER BY salary) AS rn,
               COUNT(*) OVER () AS total
        FROM emp
    )
    WHERE rn IN (
        (total + 1) / 2,
        (total + 2) / 2
    )
)
""", conn)

Median:


,median
0,30000.0


In [ ]:
# ✅ MODE
print("Mode:")
pd.read_sql_query("""
SELECT salary AS mode
FROM emp
GROUP BY salary
ORDER BY COUNT(*) DESC
LIMIT 1
""", conn)

Mode:


,mode
0,50000


**Explanation**

**Mean:**
- `AVG(salary)` directly calculates the arithmetic mean.
- (10000 + 15000 + 20000 + 20000 + 30000) / 5 = **19000**

**Median:**
- `ROW_NUMBER() OVER (ORDER BY salary)` assigns a sequential rank to each salary.
- `COUNT(*) OVER ()` gets the total number of rows.
- For **odd** count — median is the middle row: `(total + 1) / 2`
- For **even** count — median is average of two middle rows: `(total + 1) / 2` and `(total + 2) / 2`
- With 5 rows, middle row is row 3 → salary = **20000**

**Mode:**
- `GROUP BY salary` groups identical salaries.
- `ORDER BY COUNT(*) DESC` sorts by frequency — most frequent first.
- `LIMIT 1` picks the **most frequent salary** → **20000** appears twice.